# Этап 2 аугментации — парафраз писем через LLM (vLLM)

**Модель:** `Qwen/Qwen2.5-14B-Instruct-AWQ` (через vLLM)  
**Вход:** `train_after_stage1.csv`  
**Выход:** `train_after_stage2.csv`  

**Цель:** довести классы до 40 документов.

**Схема:** для каждого класса < 40 → берём документы по кругу → парафразируем батчами с запасом (x2) → фильтры → отбор. Повтор при нехватке.

## 1. Окружение и зависимости

In [1]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

In [2]:
%%capture
!pip install vllm langdetect scikit-learn

## 2. Параметры и данные

In [9]:
import re, random, gc
import numpy as np, pandas as pd
from pathlib import Path

IN_PATH  = Path("/content/train_after_stage1.csv")
OUT_PATH = Path("/content/train_after_stage2.csv")

TARGET_COUNT = 40
OVERSAMPLE   = 2
MAX_RETRIES  = 5
MIN_LENGTH   = 500
SIM_HIGH     = 0.97
SIM_LOW      = 0.50
MAX_SRC_CHARS = 4000  # обрезка исходника перед парафразом
SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    df = pd.read_csv(IN_PATH, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(IN_PATH, encoding="cp1251")

print(f"Загружено: {len(df)} документов, {df['label'].nunique()} классов")
print(f"Минимум на класс: {df['label'].value_counts().min()}")

Загружено: 1620 документов, 36 классов
Минимум на класс: 20


## 3. Загрузка LLM через vLLM

In [4]:
from vllm import LLM, SamplingParams

MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ"

llm = LLM(
    model=MODEL_NAME,
    quantization="awq",
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    enforce_eager=True,
    trust_remote_code=True,
)
print("Модель загружена через vLLM")

para_params = SamplingParams(
    temperature=0.8, top_p=0.9, top_k=50,
    max_tokens=1024, repetition_penalty=1.1,
)

INFO 05-21 14:51:05 [utils.py:240] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
WARNING 05-21 14:51:05 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 05-21 14:51:24 [model.py:568] Resolved architecture: Qwen2ForCausalLM
INFO 05-21 14:51:24 [model.py:1697] Using max model len 8192
INFO 05-21 14:51:25 [awq_marlin.py:256] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 05-21 14:51:25 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Модель загружена через vLLM


## 4. Промпт парафраза и батчевая генерация

In [5]:
SYSTEM_PROMPT = (
    "Ты переформулируешь деловое письмо на русском языке. "
    "Сохрани смысл, тему, деловой стиль и все плейсхолдеры вида "
    "[PERSON], [ORGANIZATION], [DATE_TIME] в неизменном виде. "
    "Измени формулировки, структуру предложений и порядок изложения. "
    "Пиши только текст письма — без пояснений и комментариев. "
    "Не используй Markdown-разметку."
)

PARA_PROMPT = (
    "Переформулируй следующее деловое письмо другими словами, "
    "сохранив смысл и все плейсхолдеры:\n\n{text}"
)


def llm_paraphrase_batch(texts, sampling_params):
    conversations = [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": PARA_PROMPT.format(text=t[:MAX_SRC_CHARS])}]
        for t in texts
    ]
    outputs = llm.chat(conversations, sampling_params)
    return [o.outputs[0].text.strip() if o.outputs else "" for o in outputs]

## 5. Фильтры валидации

In [6]:
from langdetect import detect, LangDetectException
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u309f\u30a0-\u30ff\uac00-\ud7af]")
PLACEHOLDER_RE = re.compile(r"\[[A-ZА-ЯЁ_]+\]")
LEAK_MARKERS = [
    "конечно,", "конечно!", "вот письмо", "вот пример", "вот переформулирован",
    "вот другой вариант", "переформулированное письмо", "как языковая модель",
    "я переформулировал", "вот текст",
]

def is_truncated(text):
    t = text.rstrip()
    return (not t) or t[-1] not in ".!?)\"»]"

def is_degenerate(text):
    words = text.lower().split()
    if len(words) < 3:
        return True
    return len(set(words)) / len(words) < 0.2

def is_prompt_leak(text):
    low = text.strip().lower()
    if any(m in low[:150] for m in LEAK_MARKERS):
        return True
    return bool(re.search(r"\*\*[^*\n]{3,}?\*\*|(?:^|\n)###?\s", text, re.MULTILINE))

def is_russian(text):
    try:
        return detect(text) == "ru"
    except LangDetectException:
        return False

def placeholders_preserved(orig, para, min_ratio=0.5):
    """Проверяет что в парафразе сохранилось достаточно плейсхолдеров."""
    orig_ph = PLACEHOLDER_RE.findall(orig)
    if not orig_ph:
        return True
    para_ph = PLACEHOLDER_RE.findall(para)
    return len(para_ph) >= len(orig_ph) * min_ratio


def validate_paraphrases(pairs, existing_texts):
    """pairs: список (оригинал, парафраз). Возвращает прошедшие парафразы."""
    seen = {t.strip().lower() for t in existing_texts}
    passed = []
    for orig, para in pairs:
        if not para or len(para.strip()) < MIN_LENGTH:
            continue
        norm = para.strip().lower()
        if norm in seen or norm == orig.strip().lower():
            continue
        if is_truncated(para) or is_degenerate(para) or is_prompt_leak(para):
            continue
        if CJK_RE.search(para) or not is_russian(para):
            continue
        if not placeholders_preserved(orig, para):
            continue
        seen.add(norm)
        passed.append(para)
    if not passed or not existing_texts:
        return passed
    # TF-IDF косинусное сходство (на символьных n-граммах)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    all_texts = existing_texts + passed
    tfidf = vec.fit_transform(all_texts)
    n_old = len(existing_texts)
    sims = cosine_similarity(tfidf[n_old:], tfidf[:n_old]).max(axis=1)
    return [p for p, s in zip(passed, sims) if SIM_LOW <= s < SIM_HIGH]

## 6. Парафраз класса (батчами)

In [7]:
def paraphrase_class(class_name, existing, n_needed):
    collected, pool = [], list(existing)
    for attempt in range(1, MAX_RETRIES + 1):
        need = n_needed - len(collected)
        if need <= 0:
            break
        batch_n = need * OVERSAMPLE + 1
        # источники для парафраза — по кругу из оригиналов
        sources = [existing[i % len(existing)] for i in range(batch_n)]
        random.shuffle(sources)
        paras = llm_paraphrase_batch(sources, para_params)
        pairs = list(zip(sources, paras))
        valid = validate_paraphrases(pairs, pool)
        take = valid[:need]
        collected.extend(take); pool.extend(take)
        print(f"  попытка {attempt}: парафразов {len(paras)}, прошло {len(valid)}, "
              f"взято {len(take)}, всего {len(collected)}/{n_needed}")
    return collected

## 7. Запуск этапа 2

In [10]:
counts = df["label"].value_counts()
classes_to_aug = counts[counts < TARGET_COUNT].sort_values()
print(f"Классов для парафраза: {len(classes_to_aug)}\n")

new_rows = []
for i, (cls, cur) in enumerate(classes_to_aug.items(), 1):
    need = TARGET_COUNT - cur
    # парафразируем только оригиналы этого класса
    existing = df[df["label"] == cls]["text"].tolist()
    print(f"[{i}/{len(classes_to_aug)}] «{cls[:40]}»: есть {cur}, нужно ещё {need}")
    gen = paraphrase_class(cls, existing, need)
    for t in gen:
        new_rows.append({"label": cls, "text": t, "source": "stage2_paraphrase"})
    tmp = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    tmp.to_csv(OUT_PATH, index=False)
    print(f"  добавлено {len(gen)}, промежуточно сохранено {len(tmp)}\n")

print(f"Этап 2 завершён. Добавлено {len(new_rows)} парафразов.")

Классов для парафраза: 26

[1/26] «Блок директора по портфелю»: есть 20, нужно ещё 20


Rendering conversations:   0%|          | 0/41 [00:00<?, ?it/s]

INFO 05-21 14:56:46 [hf.py:483] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/41 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  попытка 1: парафразов 41, прошло 29, взято 20, всего 20/20
  добавлено 20, промежуточно сохранено 1640

[2/26] «Управление землеустроительных работ»: есть 20, нужно ещё 20


...

Processed prompts:   0%|          | 0/13 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  попытка 1: парафразов 13, прошло 11, взято 6, всего 6/6
  добавлено 6, промежуточно сохранено 2077

[26/26] «Проект сервиса скважин»: есть 36, нужно ещё 4


Rendering conversations:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  попытка 1: парафразов 9, прошло 7, взято 4, всего 4/4
  добавлено 4, промежуточно сохранено 2081

Этап 2 завершён. Добавлено 461 парафразов.


## 8. Итог

In [11]:
final = pd.read_csv(OUT_PATH)
print(f"Было: {len(df)} | Стало: {len(final)} (+{len(final)-len(df)})")
print(f"\nИсточники:")
print(final["source"].value_counts())
print(f"\nМинимум на класс: {final['label'].value_counts().min()}")

from google.colab import files
files.download(str(OUT_PATH))

Было: 1620 | Стало: 2081 (+461)

Источники:
source
original             1404
stage2_paraphrase     461
stage1_llm_gen        216
Name: count, dtype: int64

Минимум на класс: 40


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>